# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Ship the rule-based queue, not the model.** ML-08/ML-09 measured both against the same held-out metric, and the transparent rule (`lost_clicks_90d`, gap × volume) beat Logistic Regression and Random Forest at every K — the models scored *below* the base rate. A queue a content editor can read and argue with, that also happens to outperform the model on the honest comparison, is the one that ships.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
pool = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)].copy()
pool["tier_median_ctr"] = pool.groupby("position_tier")["ctr"].transform("median")

# ML-06's fix: median among rows that actually measured engagement, not the raw tier median
# (which is 0.0 for top_3/striking and would make the corroboration check meaningless).
measured_median = pool[pool.engagement_rate > 0].groupby("position_tier")["engagement_rate"].median()
pool["tier_median_engagement"] = pool["position_tier"].map(measured_median)

pool["ctr_gap"] = (pool["tier_median_ctr"] - pool["ctr"]).clip(lower=0)
pool["lost_clicks_90d"] = (pool["ctr_gap"] / 100) * pool["impressions_90d"]
hi_vol_thresh = pool["impressions_90d"].quantile(0.75)

def reason_codes(row):
    r = []
    if row.ctr < row.tier_median_ctr: r.append("low_ctr_visible_page")
    if row.position_tier in ("top_3", "page_1"): r.append("leader_position_underperforming")
    if row.impressions_90d >= hi_vol_thresh: r.append("high_volume_opportunity")
    if row.engagement_rate > 0 and row.engagement_rate < row.tier_median_engagement:
        r.append("weak_engagement_confirms")
    return "|".join(r) if r else "general_ctr_review"

def human_reason(codes):
    rs = set(codes.split("|"))
    parts = []
    if "leader_position_underperforming" in rs:
        parts.append("ranks well but isn't converting that ranking into clicks")
    if "high_volume_opportunity" in rs:
        parts.append("gets enough impressions that a small CTR fix moves real click volume")
    if "weak_engagement_confirms" in rs:
        parts.append("visitors who do click aren't engaging either, a second signal pointing the same way")
    if not parts:
        parts.append("below its tier's typical CTR, worth a look when higher-priority items are cleared")
    return "; ".join(parts)

def action_and_confidence(codes):
    rs = set(codes.split("|"))
    if "high_volume_opportunity" in rs and "weak_engagement_confirms" in rs:
        return "rewrite_title_meta_and_review_page", "high"
    if "high_volume_opportunity" in rs:
        return "rewrite_title_and_meta", "medium-high"
    if "weak_engagement_confirms" in rs:
        return "review_intent_match_and_snippet", "medium"
    return "monitor_low_priority", "low"

pool["reason_codes"] = pool.apply(reason_codes, axis=1)
queue = pool[pool.ctr < pool.tier_median_ctr].sort_values("lost_clicks_90d", ascending=False).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)
queue["why_this_page"] = queue["reason_codes"].apply(human_reason)
queue["suggested_action"], queue["confidence"] = zip(*queue["reason_codes"].apply(action_and_confidence))

print(f"Ranked queue: {len(queue)} pages out of {len(pool)} eligible ({len(queue)/len(pool):.0%})\n")
for _, r in queue.head(5).iterrows():
    print(f"#{r['rank']} {r.content_id} ({r.position_tier}, {r.impressions_90d:,.0f} impr/90d)")
    print(f"   why: {r.why_this_page}")
    print(f"   do: {r.suggested_action}  (confidence: {r.confidence})\n")

Ranked queue: 5888 pages out of 12023 eligible (49%)

#1 content_36ff89c8214e (page_1, 295,097 impr/90d)
   why: ranks well but isn't converting that ranking into clicks; gets enough impressions that a small CTR fix moves real click volume; visitors who do click aren't engaging either, a second signal pointing the same way
   do: rewrite_title_meta_and_review_page  (confidence: high)

#2 content_5fe46e04994d (page_1, 517,715 impr/90d)
   why: ranks well but isn't converting that ranking into clicks; gets enough impressions that a small CTR fix moves real click volume; visitors who do click aren't engaging either, a second signal pointing the same way
   do: rewrite_title_meta_and_review_page  (confidence: high)

#3 content_c8e9d6ab9013 (page_1, 208,678 impr/90d)
   why: ranks well but isn't converting that ranking into clicks; gets enough impressions that a small CTR fix moves real click volume
   do: rewrite_title_and_meta  (confidence: medium-high)

#4 content_c84a0ab98e90 (page_1, 2

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a content editor or SEO lead uses this queue to decide *review order* among already-visible pages — which page to open first this week, not what to write. Every row keeps the reason codes and a human-readable "why," so the person can sanity-check before acting; per ML-09's claim rewrite, the score is a **directional prioritization signal, not a guaranteed click count**.

**Where it stops being valid:**
- **Outside the eligible pool.** Pages below 500 impressions/90d, ranked worse than #20, or brand-new with no CTR history yet aren't scored at all — this queue says nothing about them.
- **This snapshot only.** Built from one 90-day window in one starter export; not validated against FlyRank's full 57-brand warehouse, and CTR norms likely differ by industry/vertical, which this dataset can't distinguish (anonymized, no vertical field).
- **`weak_engagement_confirms` is a proxy, not ground truth** (ML-06) — it corroborates, it doesn't confirm; treat rows without that flag as no less real an opportunity, just less doubly-checked.
- **Not for the ML model version.** ML-08/09 showed Logistic Regression and Random Forest, trained on the same honest features, rank *worse than random* on this same metric — that version should not be shipped or trusted for prioritization as-is.

In [2]:
# Make the pool boundary and the proxy caveat checkable, not just asserted in prose.
print(f"Eligible pool definition: impressions_90d >= 500 AND 0 < avg_position <= 20")
print(f"Pages outside this pool (unscored): {len(df) - len(pool)} of {len(df)} ({(len(df)-len(pool))/len(df):.0%})")
print(f"\nShare of the ranked queue with the weak_engagement_confirms corroboration flag: "
      f"{(queue.reason_codes.str.contains('weak_engagement_confirms')).mean():.1%}")
print("-> Most of the queue does NOT carry that corroboration flag -- it's a bonus signal on "
      "some rows, not a requirement for a row to be legitimately ranked.")

Eligible pool definition: impressions_90d >= 500 AND 0 < avg_position <= 20
Pages outside this pool (unscored): 17977 of 30000 (60%)

Share of the ranked queue with the weak_engagement_confirms corroboration flag: 15.6%
-> Most of the queue does NOT carry that corroboration flag -- it's a bonus signal on some rows, not a requirement for a row to be legitimately ranked.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any row, a person checks:**
- Is the low CTR *intentional* — a deliberately plain title for a legal/policy page, or a page that isn't meant to compete on search click-through at all?
- Does `content_type` context make this a bad target (e.g. a `feedly article`, which the ML-06 audit found is missing keyword/competition data far more often — the queue may be reasoning about it with less to go on)?
- For `general_ctr_review` rows: is the gap actually small (per ML-07's tail-of-queue finding), where a fixed minimum-gap threshold would have excluded it anyway?

**Never automate:**
- Auto-publishing a rewritten title/meta description without a human editing pass — the queue flags *where* to look, never *what to write*.
- Using this queue, or any single page's rank, to evaluate a content writer's performance — the score reflects position-tier peer comparison and click behavior, not writing quality, and using it that way risks penalizing people for CTR patterns outside their control (seasonality, SERP feature changes, competitor moves).
- Shipping the Logistic Regression / Random Forest ranking in place of the rule baseline, per ML-08/09's precision@K results above.

In [3]:
# A concrete no-go check: flag rows where the queue is reasoning with less context than usual,
# so a reviewer knows to weight the recommendation more lightly.
queue["low_context_flag"] = queue["content_type"].eq("feedly article")
print(f"Rows flagged as lower-context (feedly article, sparser keyword data): "
      f"{queue['low_context_flag'].sum()} of {len(queue)} ({queue['low_context_flag'].mean():.1%})")

Rows flagged as lower-context (feedly article, sparser keyword data): 55 of 5888 (0.9%)


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Tier medians drift.** `tier_median_ctr` and `tier_median_engagement` are recomputed from whatever 90-day window is loaded — refresh them every time the underlying data refreshes (at minimum quarterly), not once and forget.
- **Base rate shift.** If the eligible-pool `is_opportunity`-style share (currently ~49%) moves sharply, the eligible-pool definition or the data source itself likely changed — investigate before trusting a new run's ranking.
- **Corroboration coverage collapses.** If the share of the queue carrying `weak_engagement_confirms` drops toward 0%, re-check for the ML-06 zero-inflation trap resurfacing (i.e. someone reverted the conditional-median fix).
- **content_type mix drift.** If `feedly article` share of the queue grows well past the pool's overall mix, the volume-weighting behavior flagged in ML-07's weak-pick review may be concentrating attention unevenly again.
- **Did the action work?** After a batch of title/meta rewrites, re-pull CTR for those specific pages next quarter — if actioned pages don't close their gap on average, that's the strongest signal to revisit the rule itself, not just re-run it.

In [4]:
# Reference values to compare future runs against.
monitoring_baseline = {
    "eligible_pool_size": len(pool),
    "is_opportunity_base_rate": round((pool.ctr < pool.tier_median_ctr).mean(), 3),
    "queue_size": len(queue),
    "weak_engagement_confirms_share": round((queue.reason_codes.str.contains("weak_engagement_confirms")).mean(), 3),
    "feedly_article_share_of_queue": round(queue["content_type"].eq("feedly article").mean(), 3),
}
for k, v in monitoring_baseline.items():
    print(f"{k:35s} {v}")

eligible_pool_size                  12023
is_opportunity_base_rate            0.49
queue_size                          5888
weak_engagement_confirms_share      0.156
feedly_article_share_of_queue       0.009


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
out_dir = Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)

export_cols = ["rank", "content_id", "client_id", "position_tier", "content_type",
               "impressions_90d", "ctr", "tier_median_ctr", "lost_clicks_90d",
               "reason_codes", "why_this_page", "suggested_action", "confidence", "low_context_flag"]
queue[export_cols].to_csv(out_dir / "action_playbook_queue.csv", index=False)

import json
with open(out_dir / "playbook_monitoring_baseline.json", "w") as f:
    json.dump(monitoring_baseline, f, indent=2)

print(f"Wrote {out_dir / 'action_playbook_queue.csv'} ({len(queue)} rows)")
print(f"Wrote {out_dir / 'playbook_monitoring_baseline.json'}")

Wrote ../outputs/action_playbook_queue.csv (5888 rows)
Wrote ../outputs/playbook_monitoring_baseline.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.